In [ ]:
# ==============================================================================
# PARALLEL DIM: EQUIPMENT
# ==============================================================================
from helpers import IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger, safe_count, generate_batch_id
import pandas as pd
from helpers.silver_transforms import transform_equipment_dimension

logger = setup_logger("parallel_dim_equipment")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

config = TableConfig(
    table_name="equipment",
    business_key="equipment_id",
    surrogate_key="equipment_key",
    watermark_column="last_update",
    scd_type=1,
    gold_table_name="dim_equipment",
    silver_transform=transform_equipment_dimension,
)

bronze_batch_id = get_latest_batch_id(spark, "equipment")
if not bronze_batch_id:
    raise ValueError("No bronze batch_id found for equipment; run bronze load first.")
logger.info(f"Using bronze batch_id for equipment: {bronze_batch_id}")

print("Row Counts (Before):")
print(f"dim_equipment: {safe_count(spark, 'dim_equipment')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))

results = pipeline.load_tables([config], force_full=False, bronze_batch_id=bronze_batch_id)
display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"dim_equipment: {safe_count(spark, 'dim_equipment')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
